# 🧪 Lab 3 : Data Lake GCS & Tables Externes vs Tables Natives BigQuery

Dans ce lab pratique, vous allez explorer le concept de **Data Lake** sur Google Cloud Storage (GCS) et sa connexion avec le **Data Warehouse** BigQuery.

Vous allez manipuler les fichiers bruts via le CLI **`gcloud storage`** dans les cellules de code exécutables ci-dessous (ou dans votre terminal Cloud Shell / VSCode), puis créer et comparer des **Tables Externes** (pointant directement sur le Data Lake) et des **Tables Natives** (stockées dans le moteur columnar Capacitor de BigQuery) via la **Console Web BigQuery**.

### 🎯 Objectifs :
1. **Administrer un Data Lake (Landing Zone)** avec `gcloud storage` : création de bucket GCS, transfert de fichiers JSON et configuration des règles de Lifecycle Management.
2. **Créer un Dataset BigQuery** `bronze_landing` dans la Console GCP.
3. **Déclarer une Table Externe** dans BigQuery pointant directement sur les fichiers JSON bruts de GCS (via la Console UI et par DDL SQL).
4. **Charger une Table Native** dans BigQuery (stockage optimisé Capacitor).
5. **Comparer & Analyser les performances** : mesurer la différence de comportement, de vitesse, d'octets scannés (*bytes scanned*) et de coût entre une Table Externe (Data Lake) et une Table Native (Data Warehouse).

---


## 0. Préparation de l'environnement CLI (Cloud Shell / VS Code Terminal)

Exécutez la cellule de code ci-dessous pour vérifier le compte actif et exporter les variables du projet GCP du cours :

In [ ]:
# 1. Vérifier le compte Google Cloud connecté
gcloud auth list

# 2. Récupérer et exporter l'ID du projet actif
export PROJECT_ID=$(gcloud config get-value project)
echo "Projet GCP actif : $PROJECT_ID"

> 💡 **Note** : Assurez-vous que le projet affiché est bien votre projet de cours (ex: `lasalle-big-data`).

---


## 1. Déploiement du Data Lake (Landing Zone) via `gcloud storage` CLI

Dans une architecture Data Lake, les données brutes sont collectées et stockées dans leur format d'origine (JSON, CSV, Parquet) au sein d'un stockage d'objets scalable comme Google Cloud Storage (GCS).

### Étape 1.1 : Créer le Bucket GCS

Les noms de buckets GCS doivent être **uniques à l'échelle mondiale**. Par convention, nous utilisons l'ID du projet GCP pour garantir l'unicité.

Exécutez la cellule ci-dessous pour créer votre bucket Data Lake dans la région `us-east1` avec le contrôle d'accès uniforme activé :

In [ ]:
# export USER_MAI
export PROJECT_ID=$(gcloud config get-value project)
export BUCKET_NAME="${PROJECT_ID}-datalake-raw"
export LOCATION="us-east1"      # définit la région

# Création du bucket via la CLI moderne gcloud storage
gcloud storage buckets create "gs://${BUCKET_NAME}" \
    --location="${LOCATION}" \
    --uniform-bucket-level-access

# Vérification de la création du bucket
gcloud storage buckets describe gs://$BUCKET_NAME

> Vous devez obtenir un descriptif du bucket.
> Regardez la ligne suivante pour obtenir le temps de rétention de la donnée en secondes : 

```yaml
soft_delete_policy:
    retentionDurationSeconds:
```


### Étape 1.2 : Ingestion de Fichiers Bruts (JSON) dans le Data Lake

Simulons l'arrivée de fichiers d'événements et de transactions dans la zone d'atterrissage (*Landing Zone*).

Exécutez la cellule de code ci-dessous pour :
1. Créer un premier fichier JSON d'événements exemple localement.
2. Téléverser le fichier dans la structure de dossiers du Data Lake (`raw/events/2026/10/`).
3. Lister le contenu du bucket GCS de façon récursive.
4. Inspecter le contenu du fichier GCS directement depuis le terminal.

In [ ]:
# 1. Créer un premier fichier d'événements JSON localement
cat <<EOF > event_sample.json
{"event_id":"e101", "user_id":"u10", "event_type":"page_view", "timestamp":"2026-10-02T10:00:00Z", "price":0.0}
{"event_id":"e102", "user_id":"u12", "event_type":"add_to_cart", "timestamp":"2026-10-02T10:05:00Z", "price":49.99}
{"event_id":"e103", "user_id":"u10", "event_type":"purchase", "timestamp":"2026-10-02T10:12:00Z", "price":49.99}
EOF

In [ ]:
# 2. Transférer le fichier vers la structure du Data Lake
gcloud storage cp event_sample.json "gs://${BUCKET_NAME}/raw/events/2026/10/event_sample.json"

In [ ]:
# 3. Vérifier le contenu du bucket de manière récursive
gcloud storage ls --recursive "gs://${BUCKET_NAME}/**"

In [ ]:
# 4. Consulter directement le contenu du fichier sur GCS depuis le terminal
gcloud storage cat "gs://${BUCKET_NAME}/raw/events/2026/10/event_sample.json"

### Étape 1.3 : Configuration du Lifecycle Management (Règle d'Archivage/Purge)

Afin de maîtriser les coûts de stockage du Data Lake, configurez une règle automatique qui supprime les fichiers temporaires de la Landing Zone après 3 jours.

Exécutez la cellule ci-dessous pour générer le fichier de politique de cycle de vie et l'appliquer au bucket GCS :

In [ ]:
export PROJECT_ID=$(gcloud config get-value project)
export BUCKET_NAME="${PROJECT_ID}-datalake-raw"

# 1. Création du fichier de règle JSON de rétention
cat <<EOF > lifecycle.json
{
  "rule": [
    {
      "action": {"type": "Delete"},
      "condition": {"age": 3}
    }
  ]
}
EOF

# 2. Appliquer la politique de cycle de vie au bucket GCS
gcloud storage buckets update "gs://${BUCKET_NAME}" --lifecycle-file=lifecycle.json

# 3. Vérification : affiche la règle active sur le bucket
gcloud storage buckets describe "gs://${BUCKET_NAME}" --format="json(lifecycle)"

---
## 2. Création du Dataset BigQuery `bronze_landing` 

Rendez-vous dans la **Console Web BigQuery** :
👉 **[https://console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)**

Vous pouvez créer le Dataset de deux façons :

### Option A : Via la Console Graphique (UI)
1. Dans le panneau de gauche **Explorateur**, cliquez sur les 3 points verticaux à côté de votre projet (ex: `lasalle-big-data`).
2. Cliquez sur **Créer un jeu de données** (*Create dataset*).
3. **ID du jeu de données** : `bronze_landing`.
4. **Emplacement des données** : `us-east1` *(doit correspondre à la région du bucket GCS !)*.
5. Cliquez sur **Créer un jeu de données**.

### Option B : Via une requête SQL DDL dans l'éditeur BigQuery
Copiez et exécutez la requête SQL suivante dans l'éditeur SQL de la console BigQuery :

In [ ]:
-- 🎯 Option B : Création DDL SQL du dataset bronze_landing
CREATE SCHEMA IF NOT EXISTS `bronze_landing` 
OPTIONS (
  location = 'us-east1' -- L'emplacement doit impérativement correspondre à la région du bucket GCS !
);

---
## 3. Data Lake -> BigQuery : Création d'une Table Externe dans la Console

Une **Table Externe** (*External Table*) permet d'interroger directement les fichiers bruts stockés sur Google Cloud Storage avec du SQL standard, **sans importer ni déplacer physiquement les données** dans BigQuery.

### Méthode 1 : Via l'interface graphique de la Console BigQuery (UI)

1. Dans l'explorateur à gauche, cliquez sur les 3 points à côté du dataset **`bronze_landing`** -> **Créer une table** (*Create table*).
2. Remplissez le formulaire :
   - **Créer une table à partir de** : `Google Cloud Storage`
   - **Sélectionner un fichier dans le bucket GCS** : Saisissez l'URI avec le wildcard `*` :
     `gs://VOTRE_PROJECT_ID-datalake-raw/raw/events/2026/10/*.json`
   - **Format de fichier** : `JSON (délimité par une nouvelle ligne)`
   - **Type de table** : 🔘 **Table externe** (*External table*)
   - **Nom de la table** : `ext_raw_events`
   - **Schéma** : Cochez ☑️ **Détection automatique** (*Auto detect*)
3. Cliquez sur le bouton bleu **Créer la table**.

--- 

### Méthode 2 : Via la requête DDL SQL dans l'éditeur de la Console

Alternativement, vous pouvez exécuter cette commande SQL directement dans la console BigQuery (pensez à remplacer `VOTRE_PROJECT_ID` par l'ID réel de votre projet) :

In [ ]:
-- Création DDL d'une Table Externe pointant sur GCS
CREATE OR REPLACE EXTERNAL TABLE `bronze_landing.ext_raw_events`
OPTIONS (
  format = 'JSON',
  uris = ['gs://lasalle-big-data-datalake-raw/raw/events/2026/10/*.json']
);

In [ ]:
-- Requête SQL sur la Table Externe (lit directement le fichier sur GCS !)
SELECT 
    event_type,
    COUNT(1) AS total_events,
    SUM(price) AS total_revenue_usd
FROM `bronze_landing.ext_raw_events`
GROUP BY event_type
ORDER BY total_events DESC;

🔍 **Observation dans la console** :
Observez les détails de la table `ext_raw_events` sous l'onglet **Détails** (*Details*) dans la console.
- La ligne **Emplacement source** (*Source URI*) indique le chemin `gs://...` sur GCS.
- **Taille de la table** : Indique *0 octet* dans le stockage BigQuery car la donnée réside physiquement sur GCS !

---

## 4. Data Warehouse : Création d'une Table Native BigQuery

Une **Table Native** (*Native Table*) importe physiquement les données au sein du moteur de stockage distribué propriétaire de BigQuery (**Capacitor**). Les données sont encodées, compressées et structurées en colonnes pour des performances analytiques maximales.

### Méthode 1 : DDL SQL `CREATE TABLE AS SELECT` dans la Console
Exécutez cette requête SQL dans la console BigQuery pour ingérer les données de la Table Externe vers une Table Native :

In [ ]:
-- Importation physique dans le stockage natif Capacitor (Data Warehouse)
CREATE OR REPLACE TABLE `bronze_landing.native_raw_events` AS
SELECT 
    event_id,
    user_id,
    event_type,
    PARSE_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ', timestamp) AS event_timestamp,
    price
FROM `bronze_landing.ext_raw_events`;


## 5. Comparatif & Benchmark : Table Externe (Data Lake) vs Table Native (Data Warehouse)

Analysons les différences clés entre les deux modes d'accès dans la console.

### Test 1 : Interrogation de la Table Native
Collez et exécutez la requête sur la table native :

```sql
-- Requête analytique sur la Table Native
SELECT 
    event_type,
    COUNT(1) AS total_events,
    SUM(price) AS total_revenue_usd
FROM `bronze_landing.native_raw_events`
GROUP BY event_type;
```

### ⚖️ Tableau Comparatif Synthétique

| Critère de comparaison | Table Externe (Data Lake) | Table Native (Data Warehouse) |
| :--- | :--- | :--- |
| **Emplacement des données** | Fichiers bruts sur Google Cloud Storage (GCS) | Stockage distribué BigQuery (Capacitor) |
| **Format physique** | JSON, CSV, Avro, Parquet brut | Format colonnaire propriétaire optimisé |
| **Temps de lecture / Latence** | Moins rapide (overhead de parsing à la volée du JSON/CSV) | Extrêmement rapide (indexé, vectorisé) |
| **Coût de stockage** | Facturation GCS Standard (~0.02 $/Go/mois) | Facturation BigQuery Storage (~0.02 $/Go/mois) |
| **Coût des requêtes (Octets scannés)** | Scanne le fichier JSON complet à chaque requête | Ne scanne **QUE** les colonnes demandées |
| **Modifications DML (`UPDATE`/`DELETE`)** | ❌ Impossible (Lecture seule) | ✅ Supporté avec gestion ACID |
| **Partitionnement & Clustering** | Limité (Partitionnement Hive par dossiers GCS) | ✅ Optimisation avancée par partition/cluster |
| **Cas d'usage principal** | Exploration ad-hoc, Landing zone, Données fraîches | Data Warehouse principal, Dashboards production |

---


### Méthode 2 : Import direct via l'UI Console (*Charger une table* / `bq load` équivalent)
Vous pouvez aussi charger des données directement de GCS vers une table native :
1. Cliquez sur le dataset `bronze_landing` -> **Créer une table**.
2. Source : `Google Cloud Storage` -> Fichier : `gs://VOTRE_PROJECT_ID-datalake-raw/raw/events/2026/10/*.json`.
3. Format : `JSON (délimité par une nouvelle ligne)`.
4. Type de table : 🔘 **Table native** (*Native table*).
5. Nom de la table : `native_raw_events_ui`.
6. Schéma : ☑️ **Détection automatique**.
7. Cliquez sur **Créer la table**.

---

## 6. Exercice Pratique : Simuler l'arrivée de données en temps réel 🏋️

L'un des avantages majeurs des **Tables Externes** est qu'elles reflètent immédiatement tout nouveau fichier déposé dans le bucket GCS sans aucune action de rechargement dans BigQuery !

### Étape 6.1 : Ajouter un 2ème fichier dans GCS via la CLI
Exécutez la cellule de code ci-dessous :

In [ ]:
export PROJECT_ID=$(gcloud config get-value project)
export BUCKET_NAME="${PROJECT_ID}-datalake-raw"

In [ ]:
# Créer un second fichier de logs JSON localement
cat <<EOF > event_sample_part2.json
{"event_id":"e104", "user_id":"u15", "event_type":"page_view", "timestamp":"2026-10-02T11:00:00Z", "price":0.0}
{"event_id":"e105", "user_id":"u15", "event_type":"purchase", "timestamp":"2026-10-02T11:05:00Z", "price":129.50}
EOF

In [ ]:
# Déposer le fichier dans le dossier GCS du Data Lake
gcloud storage cp event_sample_part2.json "gs://${BUCKET_NAME}/raw/events/2026/10/event_sample_part2.json"

### Étape 6.2 : Interroger la Table Externe vs Table Native dans la Console BigQuery

1. Re-exécutez la requête sur la **Table Externe** dans la console BigQuery :


In [ ]:
SELECT COUNT(1) AS total_lignes_externe FROM `bronze_landing.ext_raw_events`;


👉 **Résultat attendu** : Le nombre total de lignes est désormais **5** (les 3 lignes du 1er fichier + les 2 lignes du 2ème fichier) ! La table externe s'est mise à jour automatiquement.

2. Re-exécutez la même requête sur la **Table Native** :

In [ ]:

SELECT COUNT(1) AS total_lignes_native FROM `bronze_landing.native_raw_events`;


👉 **Résultat attendu** : Le nombre de lignes reste à **3**. La table native nécessite un nouveau `INSERT` ou `bq load` pour intégrer les nouvelles données de la Landing Zone.

---

## ✅ Checkpoint de Validation du Lab

- [ ] J'ai créé un Bucket GCS `gs://[PROJECT_ID]-datalake-raw` et configuré une règle de Lifecycle Management avec `gcloud storage`.
- [ ] J'ai créé le Dataset BigQuery `bronze_landing` en région `us-east1` (via l'UI Console ou par DDL SQL `CREATE SCHEMA`).
- [ ] J'ai créé une Table Externe `ext_raw_events` pointant directement sur les fichiers JSON du Data Lake via l'UI Console et/ou DDL SQL.
- [ ] J'ai créé une Table Native `native_raw_events` stockée au format Capacitor dans BigQuery.
- [ ] J'ai vérifié que l'ajout d'un fichier sur GCS est immédiatement visible dans la Table Externe, mais pas dans la Table Native.
- [ ] J'ai compris les compromis entre Data Lake (souplesse, données brutes, rafraîchissement instantané) et Data Warehouse (haute performance, stockage colonnaire, DML).